In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gmean

<h3>Import Raw FM23 Data</h3><br>
Load the Football Manager 2023 player database containing player information and attribute ratings.<br>
<b>From fm2023.csv</b> <br>
Name, Club, Division, Nationality, Age, Position, Market value, Wage, etc.<br>
Technical, Mental, Physical, and Goalkeeping attributes<br>
<b>From weights.csv</b> <br>
Position weightings used by Football Manager <br>

In [2]:
# Import raw data, fill blanks and remove leading/lagging spaces
df = pd.read_csv('fm2023.csv')
weights_df = pd.read_csv('weights.csv', index_col=0)
weights_df = weights_df.fillna(0)
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include=["object", "string"]):
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
df.head()

,UID,Inf,Name,DoB,Nat,Division,Club,Based,Preferred Foot,Right Foot,...,Ref,TRO,Sta,Str,Tck,Tea,Tec,Thr,Vis,Wor
0,831079,,Giorgio Chiellini,14/8/1984 (37 years old),ITA,Major League Soccer,LAFC,U.S.A. (MLS),Left Only,Weak,...,1,2,10,14,18,17,8,4,7,15
1,2105510,,Hulk,25/7/1986 (35 years old),BRA,Brazilian National First Division,ATM,Brazil (First Division),Left,Reasonable,...,2,1,15,18,8,8,15,3,14,11
2,78000335,,Luis Suárez,24/1/1987 (35 years old),URU,Uruguayan First Division,Nacional,Uruguay (First Division),Right,Very Strong,...,1,3,10,14,9,13,15,3,15,9
3,962988,Ret,Gonzalo Higuaín,10/12/1987 (34 years old),ARG,Major League Soccer,Inter Miami,U.S.A. (MLS),Right,Very Strong,...,1,3,10,16,8,10,16,1,14,10
4,19007730,,Alex Teixeira,6/1/1990 (32 years old),BRA,Brazilian National Second Division,VDG,Brazil (Second Division),Right,Very Strong,...,1,3,11,7,5,10,16,2,11,10


<h3>Clean and standardize data</h3>
Remove leading and trailing spaces (str.strip())<br>
Convert currency strings into numeric values<br>
Convert attribute columns to numeric data types<br>
Fill missing values with zero<br>

In [3]:
# Clean up non-numerical transfer values, change to Min and Max
s = df['Transfer Value'].replace("Not for Sale", "€0 - €0")
vals = s.str.split(r"\s*-\s*", expand=True)
for col, new_col in zip([0, 1], ["Min Value", "Max Value"]):
    num = vals[col].str.replace("€", "")
    df[new_col] = (
        num.str.extract(r"([\d.]+)")[0].astype(float) * 
        np.where(num.str.endswith("M"), 1_000_000, np.where(num.str.endswith("K"), 1_000, 1))
    )
    col_to_move = df.pop(new_col)
    df.insert(16, new_col, col_to_move)
df.head(20)

,UID,Inf,Name,DoB,Nat,Division,Club,Based,Preferred Foot,Right Foot,...,Ref,TRO,Sta,Str,Tck,Tea,Tec,Thr,Vis,Wor
0,831079,,Giorgio Chiellini,14/8/1984 (37 years old),ITA,Major League Soccer,LAFC,U.S.A. (MLS),Left Only,Weak,...,1,2,10,14,18,17,8,4,7,15
1,2105510,,Hulk,25/7/1986 (35 years old),BRA,Brazilian National First Division,ATM,Brazil (First Division),Left,Reasonable,...,2,1,15,18,8,8,15,3,14,11
2,78000335,,Luis Suárez,24/1/1987 (35 years old),URU,Uruguayan First Division,Nacional,Uruguay (First Division),Right,Very Strong,...,1,3,10,14,9,13,15,3,15,9
3,962988,Ret,Gonzalo Higuaín,10/12/1987 (34 years old),ARG,Major League Soccer,Inter Miami,U.S.A. (MLS),Right,Very Strong,...,1,3,10,16,8,10,16,1,14,10
4,19007730,,Alex Teixeira,6/1/1990 (32 years old),BRA,Brazilian National Second Division,VDG,Brazil (Second Division),Right,Very Strong,...,1,3,11,7,5,10,16,2,11,10
5,8825061,,Fernandinho,4/5/1985 (37 years old),BRA,Brazilian National First Division,ATP,Brazil (First Division),Right,Very Strong,...,2,1,11,13,15,17,16,1,12,17
6,315542,,Daniel Alves,6/5/1983 (39 years old),BRA,Mexican First Division,Pumas,Mexico (First Division),Right,Very Strong,...,3,1,13,11,12,14,16,1,15,15
7,5132312,,Gareth Bale,16/7/1989 (32 years old),WAL,Major League Soccer,LAFC,U.S.A. (MLS),Left,Reasonable,...,4,1,11,14,9,7,16,1,13,7
8,51000001,,Carlos Vela,1/3/1989 (33 years old),MEX,Major League Soccer,LAFC,U.S.A. (MLS),Left Only,Weak,...,3,3,10,7,5,9,17,2,15,7
9,43043933,,Lorenzo Insigne,4/6/1991 (31 years old),ITA,Major League Soccer,Toronto FC,U.S.A. (MLS),Right,Very Strong,...,3,2,14,6,6,14,17,2,16,13


<h3>Calculate Player Ratings</h3>
For each position, multiply player attributes by the corresponding FM attribute weightings, sum and normalize.

In [4]:
# Calculate ratings per position from raw attributes data using the weightings in Football Manager
player_attrs = set(df.columns)
weight_attrs = set(weights_df.columns)

attribute_cols = weights_df.columns
player_attributes = df[attribute_cols]
weighted_sum = player_attributes.dot(weights_df.T)
weight_totals = weights_df.sum(axis=1)
ratings = weighted_sum.div(weight_totals, axis=1)
ratings = 20*(ratings - 6)
print(ratings.head(10))

          GK          DC         DRL          DM        WBRL          MC  \
0  64.859813  169.450549  138.974359  133.181818  114.933333  110.444444   
1  59.439252  115.384615  134.358974  137.500000  148.266667  148.222222   
2  47.289720  113.846154  120.256410  127.272727  122.933333  138.000000   
3  38.130841  112.747253  118.461538  127.045455  122.400000  136.222222   
4  30.093458   78.901099   96.923077  103.863636  109.333333  117.555556   
5  60.000000  151.428571  149.743590  154.545455  145.866667  152.000000   
6  46.728972  122.857143  137.692308  143.409091  145.866667  150.444444   
7  37.943925  105.274725  112.307692  114.090909  117.600000  124.222222   
8  42.429907   81.318681  104.615385  110.454545  121.066667  128.444444   
9  49.532710   84.835165  120.769231  129.090909  140.533333  147.111111   

          MRL         AMC        AMRL          ST  
0   93.589744   94.545455   86.590909   94.953271  
1  158.461538  156.818182  159.318182  154.205607  
2  138.

A player's rating (Current Ability) is the highest out of his ratings per position.

In [5]:
# Best position = select position with highest rating from positional ratings
df["Best_Position"] = ratings.idxmax(axis=1)
# Best rating = select highest rating
df["Current Ability"] = ratings.max(axis=1)
df["Value"] = gmean(df[["Min Value", "Max Value"]], axis=1)
print(df.head())

        UID  Inf               Name                        DoB  Nat  \
0    831079       Giorgio Chiellini   14/8/1984 (37 years old)  ITA   
1   2105510                    Hulk   25/7/1986 (35 years old)  BRA   
2  78000335             Luis Suárez   24/1/1987 (35 years old)  URU   
3    962988  Ret    Gonzalo Higuaín  10/12/1987 (34 years old)  ARG   
4  19007730           Alex Teixeira    6/1/1990 (32 years old)  BRA   

                             Division         Club                     Based  \
0                 Major League Soccer         LAFC              U.S.A. (MLS)   
1   Brazilian National First Division          ATM   Brazil (First Division)   
2            Uruguayan First Division     Nacional  Uruguay (First Division)   
3                 Major League Soccer  Inter Miami              U.S.A. (MLS)   
4  Brazilian National Second Division          VDG  Brazil (Second Division)   

  Preferred Foot   Right Foot  ... Str Tck Tea Tec  Thr Vis  Wor  \
0      Left Only        

<h3>League Filters</h3>
Ensure league comparisons are based on sufficiently complete squads.<br>
Retain only leagues where every club contains at least 16 players.<br>
This removes leagues with incomplete databases that may distort squad rankings and league benchmarks.

In [6]:
# Count players per club, only leagues with all clubs having 16 players minimum will be retained
club_sizes = (df.groupby(["Division", "Club"]).size().reset_index(name="Players"))
valid_leagues = (club_sizes.groupby("Division")["Players"].min().loc[lambda x: x >= 16].index)
df = df[df["Division"].isin(valid_leagues)].copy()
print(f"Leagues retained: {len(valid_leagues)}")
print(f"Players retained: {len(df)}")


Leagues retained: 187
Players retained: 94455


<h3>Rank Players Within Each Club</h3>
Identify player status (Star, Starter, Rotation, Backup, Fringe) within each squad, ranked within their club by Current Ability.<br>
Star	1–3
Starter	4–11
Rotation	12–18
Backup	19–25
Fringe	26+

In [7]:

# Rank players within each club
df["SquadRank"] = (df.groupby("Club")["Current Ability"].rank(method="first", ascending=False))
conditions = [
    df["SquadRank"] <= 3,
    df["SquadRank"].between(4, 11),
    df["SquadRank"].between(12, 18),
    df["SquadRank"].between(19, 25),
    df["SquadRank"] >= 26
]
choices = [
    "Star",
    "Starter",
    "Rotation",
    "Backup",
    "Fringe"
]
df["SquadRole"] = np.select(conditions, choices, default="Unknown")
# League strength data
league_strength = (df.groupby(["Division", "SquadRole"])["Current Ability"].median().unstack().sort_values(by="Starter", ascending=False))
league_strength.to_csv('league_strength.csv')
league_strength.head()

SquadRole,Backup,Fringe,Rotation,Star,Starter
Division,,,,,
English Premier Division,133.297952,73.015734,143.735826,162.972600,152.818071
Spanish First Division,128.636364,77.943925,138.926432,156.818182,146.478521
Italian Serie A,128.409091,68.084626,136.647103,154.375531,144.963370
Bundesliga,124.008071,75.090272,135.134196,151.025641,143.186813
Ligue 1 Uber Eats,115.000000,72.967033,130.400000,149.282051,138.977273


<h3>Generate League Strength Benchmarks</h3>
Measure player quality levels within each league.
Group players by <b>division</b> and <b>squad role</b>, and then calculate league level <b>medians</b> for Star players, Starters, Rotation players, Backups, and Fringe players

In [11]:
top18 = df[df["SquadRank"] <= 18].copy()
role_means = (top18.groupby(["Division", "SquadRole"])["Current Ability"].mean().unstack())

league_stats = (top18.groupby("Division")["Current Ability"].agg(Total_Mean="mean", Total_SD="std"))
league_z = role_means.join(league_stats)
league_z["Star_Z"] = ((league_z["Star"] - league_z["Total_Mean"])/ league_z["Total_SD"])
league_z["Starter_Z"] = ((league_z["Starter"] - league_z["Total_Mean"])/ league_z["Total_SD"])
league_z["Rotation_Z"] = ((league_z["Rotation"] - league_z["Total_Mean"])/ league_z["Total_SD"])
league_z = league_z[[
        "Star",
        "Starter",
        "Rotation",
        "Total_Mean",
        "Total_SD",
        "Star_Z",
        "Starter_Z",
        "Rotation_Z"
    ]].sort_values(by="Starter", ascending=False)
print(league_z)
league_z.to_csv("league_zscores.csv")

                                Star     Starter    Rotation  Total_Mean  \
Division                                                                   
English Premier Division  167.044411  155.826538  146.737786  154.161669   
Spanish First Division    159.703387  149.919609  142.144759  148.526686   
Italian Serie A           157.338775  148.607120  140.714460  146.993028   
Bundesliga                153.474564  144.321799  136.429009  142.777842   
Ligue 1 Uber Eats         149.864176  141.186319  131.798652  139.001878   
...                              ...         ...         ...         ...   
Sanma Premier Division     47.281960   40.884782   33.065423   38.910116   
Malampa Premier Division   49.241008   40.227045   35.923412   40.055737   
JD Cymru South             55.123266   38.067673   25.552257   36.043165   
JD Cymru North             50.309177   36.617579   27.698728   35.431070   
Bruneian Premier League    38.954009   23.583709   13.282873   22.139545   

           

In [ ]:
print(league_z[["Star_Z", "Starter_Z", "Rotation_Z"]].agg(["mean", "std"]))

        Star_Z  Starter_Z  Rotation_Z
mean  1.266395   0.203206   -0.778802
std   0.179545   0.068519    0.106954
